In [ ]:
cpp
#include <gtest/gtest.h>
#include <vector>
#include <algorithm>
#include <stdexcept>

// Шаблонный интерфейс для структуры данных "Очередь"
template <typename T>
class QueueInterface {
public:
    virtual ~QueueInterface() = default;
    virtual void push(const T& element) = 0;
    virtual void pop() = 0;
    virtual T front() const = 0;
    virtual bool empty() const = 0;
    virtual size_t size() const = 0;
};

// Шаблонный интерфейс для структуры данных "Куча" (приоритетная очередь)
template <typename T>
class HeapInterface {
public:
    virtual ~HeapInterface() = default;
    virtual void push(const T& element) = 0;
    virtual void pop() = 0;
    virtual T top() const = 0;
    virtual bool empty() const = 0;
    virtual size_t size() const = 0;
};

// Шаблонный интерфейс для структуры данных "Бинарное дерево поиска"
template <typename T>
class BinaryTreeInterface {
public:
    virtual ~BinaryTreeInterface() = default;
    virtual void push(const T& element) = 0;
    virtual void pop(const T& element) = 0;
    virtual bool search(const T& element) const = 0;
    virtual bool empty() const = 0;
    virtual size_t size() const = 0;
};

// Реализация демонстрационных классов-заглушек для проверки работоспособности тестов
template <typename T>
class MockQueue : public QueueInterface<T> {
private:
    std::vector<T> container;
public:
    void push(const T& element) override {
        container.push_back(element);
    }
    void pop() override {
        if (empty()) throw std::underflow_error("Очередь пуста");
        container.erase(container.begin());
    }
    T front() const override {
        if (empty()) throw std::underflow_error("Очередь пуста");
        return container.front();
    }
    bool empty() const override { return container.empty(); }
    size_t size() const override { return container.size(); }
};

template <typename T>
class MockHeap : public HeapInterface<T> {
private:
    std::vector<T> container;
public:
    void push(const T& element) override {
        container.push_back(element);
        std::push_heap(container.begin(), container.end());
    }
    void pop() override {
        if (empty()) throw std::underflow_error("Куча пуста");
        std::pop_heap(container.begin(), container.end());
        container.pop_back();
    }
    T top() const override {
        if (empty()) throw std::underflow_error("Куча пуста");
        return container.front();
    }
    bool empty() const override { return container.empty(); }
    size_t size() const override { return container.size(); }
};

template <typename T>
class MockBinaryTree : public BinaryTreeInterface<T> {
private:
    std::vector<T> container;
public:
    void push(const T& element) override {
        if (!search(element)) {
            container.push_back(element);
            std::sort(container.begin(), container.end());
        }
    }
    void pop(const T& element) override {
        auto it = std::find(container.begin(), container.end(), element);
        if (it != container.end()) {
            container.erase(it);
        }
    }
    bool search(const T& element) const override {
        return std::binary_search(container.begin(), container.end(), element);
    }
    bool empty() const override { return container.empty(); }
    size_t size() const override { return container.size(); }
};

// Раздел юнит-тестирования интерфейса Очереди
TEST(QueueInterfaceTest, MainOperations) {
    MockQueue<int> test_queue;
    EXPECT_TRUE(test_queue.empty());
    
    test_queue.push(10);
    test_queue.push(20);
    test_queue.push(30);
    
    EXPECT_EQ(test_queue.size(), 3);
    EXPECT_FALSE(test_queue.empty());
    EXPECT_EQ(test_queue.front(), 10);
    
    test_queue.pop();
    EXPECT_EQ(test_queue.front(), 20);
    
    test_queue.pop();
    test_queue.pop();
    EXPECT_TRUE(test_queue.empty());
}

// Раздел юнит-тестирования интерфейса Кучи
TEST(HeapInterfaceTest, OrderingOperations) {
    MockHeap<int> test_heap;
    EXPECT_TRUE(test_heap.empty());
    
    test_heap.push(5);
    test_heap.push(45);
    test_heap.push(15);
    
    EXPECT_EQ(test_heap.top(), 45);
    
    test_heap.pop();
    EXPECT_EQ(test_heap.top(), 15);
}

// Раздел юнит-тестирования интерфейса Бинарного Дерева
TEST(BinaryTreeInterfaceTest, SearchAndModification) {
    MockBinaryTree<int> test_tree;
    EXPECT_TRUE(test_tree.empty());
    
    test_tree.push(100);
    test_tree.push(50);
    test_tree.push(150);
    
    EXPECT_TRUE(test_tree.search(50));
    EXPECT_FALSE(test_tree.search(200));
    
    test_tree.pop(50);
    EXPECT_FALSE(test_tree.search(50));
    EXPECT_EQ(test_tree.size(), 2);
}

int main(int argc, char **argv) {
    ::testing::InitGoogleTest(&argc, argv);
    return RUN_ALL_TESTS();
}
